# SGLang 常见报错及快速解决办法

**定位：** 汇总 SGLang 使用过程中最常见的错误及其快速排查与修复方法，帮助开发者快速定位问题。

**服务端启动命令（默认示例）：**

```bash
python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 30000
```

> 本 Notebook 为原创示例代码，可自由用于学习、教学及发布到 Gitee 等平台

---

## 硬件与环境要求

| 项目 | 最低要求 |
|------|----------|
| GPU | NVIDIA GPU（支持 CUDA） |
| 显存（VRAM） | ≥ 6 GB（推荐 8 GB 以上） |
| 驱动 | NVIDIA Driver ≥ 525 |
| CUDA | CUDA ≥ 12.1 |
| Python | ≥ 3.9 |
| 操作系统 | Linux / WSL2 推荐 |

## 依赖安装

```bash
pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"
```

---

## 使用指南

1. 运行「环境检查」单元格，确认基础环境
2. 运行「环境诊断工具」单元格，加载诊断函数
3. 根据遇到的错误，跳转到对应章节查看解决办法
4. 运行「一键诊断脚本」获取完整环境报告

---

In [ ]:
# ============================================================
# 环境检查 —— 确认 Python、PyTorch、CUDA、SGLang 是否就绪
# ============================================================

import sys  # 导入系统模块

print("Python 版本：", sys.version)  # 打印 Python 版本

# ---------- 检查 PyTorch ----------
try:  # 尝试导入 PyTorch
    import torch  # 导入 PyTorch 库
    print("PyTorch 版本：", torch.__version__)  # 打印 PyTorch 版本
    print("CUDA 是否可用：", torch.cuda.is_available())  # 检查 CUDA 是否可用
    if torch.cuda.is_available():  # 如果 CUDA 可用
        print("GPU 名称：", torch.cuda.get_device_name(0))  # 打印 GPU 名称
except ImportError:  # 如果 PyTorch 未安装
    print("PyTorch 未安装")  # 提示未安装

# ---------- 检查 SGLang ----------
try:  # 尝试导入 SGLang
    import sglang  # 导入 sglang
    print("SGLang 版本：", sglang.__version__)  # 打印版本
except ImportError:  # 如果未安装
    print("SGLang 未安装")  # 提示未安装

print("\n环境检查完成")  # 打印完成提示

In [ ]:
# ============================================================
# 可选安装 —— 如依赖缺失则取消注释执行
# ============================================================

# !pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"  # 安装 SGLang 全部依赖及 OpenAI 客户端和 requests 库

## SGLang 使用中的常见问题

本 Notebook 覆盖以下 6 种最常见的错误：

1. **CUDA out of memory** — 显存不足
2. **Address already in use** — 端口冲突
3. **ModuleNotFoundError: No module named 'sglang'** — 未安装 SGLang
4. **CUDA driver version is insufficient** — 驱动版本过旧
5. **Model not found / not supported** — 模型路径或架构错误
6. **Connection refused** — 无法连接 API

每种错误都提供了**原因分析**和**诊断代码**。

---

## 环境诊断工具

以下单元格定义了一组通用诊断函数，后续章节会反复使用。

In [ ]:
# ============================================================
# 环境诊断工具 —— 定义通用诊断函数供后续使用
# ============================================================

import sys  # 导入系统模块
import subprocess  # 导入子进程模块
import socket  # 导入网络套接字模块

def check_python_version():  # 定义检查 Python 版本的函数
    """检查 Python 版本是否满足要求。"""
    version = sys.version_info  # 获取 Python 版本信息元组
    ok = version.major >= 3 and version.minor >= 9  # 判断是否 >= 3.9
    status = "通过" if ok else "未通过"  # 设置通过/失败标记
    print(f"  [{status}] Python 版本: {sys.version.split()[0]} (需要 >= 3.9)")  # 打印检测结果
    return ok  # 返回是否通过

def check_torch_cuda():  # 定义检查 PyTorch 和 CUDA 的函数
    """检查 PyTorch 和 CUDA 可用性。"""
    try:  # 尝试导入 PyTorch
        import torch  # 导入 PyTorch
        print(f"  [通过] PyTorch 版本: {torch.__version__}")  # 打印 PyTorch 版本
        if torch.cuda.is_available():  # 如果 CUDA 可用
            gpu_name = torch.cuda.get_device_name(0)  # 获取 GPU 名称
            vram = torch.cuda.get_device_properties(0).total_mem / 1024**3  # 计算总显存（GB）
            print(f"  [通过] CUDA 可用: {gpu_name} ({vram:.1f} GB)")  # 打印 GPU 信息
            return True  # 返回通过
        else:  # 如果 CUDA 不可用
            print("  [未通过] CUDA 不可用")  # 打印不可用提示
            return False  # 返回失败
    except ImportError:  # 如果 PyTorch 未安装
        print("  [未通过] PyTorch 未安装")  # 打印未安装提示
        return False  # 返回失败

def check_sglang_installed():  # 定义检查 SGLang 安装状态的函数
    """检查 SGLang 是否已安装。"""
    try:  # 尝试导入 sglang
        import sglang  # 导入 sglang
        print(f"  [通过] SGLang 版本: {sglang.__version__}")  # 打印版本号
        return True  # 返回通过
    except ImportError:  # 如果未安装
        print("  [未通过] SGLang 未安装")  # 打印未安装提示
        return False  # 返回失败

def check_port_in_use(port=30000):  # 定义检查端口占用的函数
    """检查指定端口是否被占用。"""
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)  # 创建 TCP 套接字
    try:  # 尝试连接端口
        sock.settimeout(1)  # 设置连接超时为 1 秒
        result = sock.connect_ex(('127.0.0.1', port))  # 尝试连接指定端口
        if result == 0:  # 如果连接成功，说明端口被占用
            print(f"  [警告] 端口 {port} 已被占用")  # 打印端口占用警告
            return True  # 返回 True 表示被占用
        else:  # 如果连接失败，说明端口空闲
            print(f"  [通过] 端口 {port} 空闲可用")  # 打印端口空闲提示
            return False  # 返回 False 表示空闲
    finally:  # 无论结果如何
        sock.close()  # 关闭套接字

def check_gpu_memory():  # 定义检查 GPU 显存的函数
    """检查 GPU 显存使用情况。"""
    try:  # 尝试使用 PyTorch 查询显存
        import torch  # 导入 PyTorch
        if torch.cuda.is_available():  # 如果 CUDA 可用
            total = torch.cuda.get_device_properties(0).total_mem / 1024**3  # 计算总显存
            reserved = torch.cuda.memory_reserved(0) / 1024**3  # 计算已预留显存
            print(f"  总显存: {total:.2f} GB")  # 打印总显存
            print(f"  已预留: {reserved:.2f} GB")  # 打印已预留显存
            print(f"  可用约: {total - reserved:.2f} GB")  # 打印可用显存
        else:  # CUDA 不可用
            print("  CUDA 不可用，无法检查显存")  # 打印提示
    except ImportError:  # PyTorch 未安装
        print("  PyTorch 未安装，无法检查显存")  # 打印提示

print("诊断函数已加载")  # 打印加载完成提示
print("可用函数：check_python_version(), check_torch_cuda(), check_sglang_installed(), check_port_in_use(), check_gpu_memory()")  # 打印可用函数列表

---

## 错误 1：CUDA out of memory

**报错信息：**
```
torch.cuda.OutOfMemoryError: CUDA out of memory.
```

**原因分析：**
- 模型权重 + KV Cache + 运行时张量超出 GPU 可用显存
- 可能有其他进程占用了显存

**解决方案：**
1. 使用 `--mem-fraction-static 0.7` 降低 KV Cache 预分配
2. 使用 `--context-length 2048` 限制上下文长度
3. 使用量化模型（AWQ/GPTQ）
4. 选择更小的模型

In [ ]:
# ============================================================
# 错误 1 诊断 —— CUDA out of memory 排查
# ============================================================

import subprocess  # 导入子进程模块

print("===== CUDA OOM 诊断 =====")  # 打印诊断标题
print()  # 打印空行

# ---------- 检查当前显存使用 ----------
print("【1】当前 GPU 显存使用情况：")  # 打印步骤标题
check_gpu_memory()  # 调用显存检查函数

# ---------- 检查其他 GPU 进程 ----------
print("\n【2】当前 GPU 上运行的进程：")  # 打印步骤标题
try:  # 尝试执行 nvidia-smi 查看进程
    result = subprocess.run(  # 运行 nvidia-smi 命令
        ["nvidia-smi", "--query-compute-apps=pid,name,used_memory",  # 查询 GPU 上的计算进程
         "--format=csv,noheader"],  # CSV 格式无表头
        capture_output=True, text=True  # 捕获输出为文本
    )
    if result.stdout.strip():  # 如果有输出内容
        print(f"  发现以下进程正在使用 GPU：")  # 打印发现进程提示
        for line in result.stdout.strip().split("\n"):  # 遍历每行输出
            print(f"    {line}")  # 打印进程信息
    else:  # 如果没有输出
        print("  未发现 GPU 上运行的计算进程")  # 打印无进程提示
except FileNotFoundError:  # 如果 nvidia-smi 不存在
    print("  nvidia-smi 未找到")  # 打印提示

# ---------- 给出建议启动命令 ----------
print("\n【3】建议使用以下优化参数启动：")  # 打印建议标题
optimized_cmd = (  # 构建优化后的启动命令字符串
    "python -m sglang.launch_server "
    "--model-path Qwen/Qwen2.5-0.5B-Instruct "
    "--host 127.0.0.1 --port 30000 "
    "--mem-fraction-static 0.7 "
    "--context-length 2048"
)
print(f"  {optimized_cmd}")  # 打印优化命令

---

## 错误 2：Address already in use（端口冲突）

**报错信息：**
```
OSError: [Errno 98] Address already in use
```

**原因分析：**
- 端口 30000 已被另一个 SGLang 实例或其他服务占用
- 上一次启动的服务未正确关闭

**解决方案：**
1. 找到并终止占用端口的进程
2. 或者换一个端口启动

In [ ]:
# ============================================================
# 错误 2 诊断 —— Address already in use 排查
# ============================================================

import subprocess  # 导入子进程模块
import platform  # 导入平台检测模块

target_port = 30000  # 设置目标检查端口

print(f"===== 端口 {target_port} 冲突诊断 =====")  # 打印诊断标题
print()  # 打印空行

# ---------- 检查端口是否被占用 ----------
print("【1】端口占用检查：")  # 打印步骤标题
check_port_in_use(target_port)  # 调用端口检查函数

# ---------- 查找占用端口的进程 ----------
print(f"\n【2】查找占用端口 {target_port} 的进程：")  # 打印步骤标题
os_type = platform.system()  # 获取当前操作系统类型

if os_type == "Linux" or os_type == "Darwin":  # 如果是 Linux 或 macOS
    try:  # 尝试使用 lsof 命令查找
        result = subprocess.run(  # 执行 lsof 命令
            ["lsof", "-i", f":{target_port}"],  # 查找使用指定端口的进程
            capture_output=True, text=True  # 捕获输出为文本
        )
        if result.stdout.strip():  # 如果有输出
            print(f"  发现以下进程占用端口 {target_port}：")  # 打印发现提示
            for line in result.stdout.strip().split("\n"):  # 遍历输出每行
                print(f"    {line}")  # 打印进程信息
        else:  # 如果没有输出
            print(f"  未发现占用端口 {target_port} 的进程")  # 打印无进程提示
    except FileNotFoundError:  # 如果 lsof 不存在
        print("  lsof 命令不可用")  # 打印提示
else:  # 如果是 Windows 系统
    try:  # 尝试使用 netstat
        result = subprocess.run(  # 执行 netstat 命令
            ["netstat", "-ano"],  # 列出所有连接
            capture_output=True, text=True  # 捕获输出
        )
        for line in result.stdout.split("\n"):  # 遍历每行
            if str(target_port) in line:  # 如果包含目标端口
                print(f"    {line.strip()}")  # 打印匹配行
    except Exception as e:  # 如果发生异常
        print(f"  检查失败: {e}")  # 打印错误信息

# ---------- 终止进程的命令提示 ----------
print(f"\n【3】如需终止占用端口的进程：")  # 打印终止方法标题
print(f"  Linux/macOS: kill -9 $(lsof -t -i:{target_port})")  # 打印终止命令
print(f"  或者换用其他端口启动：--port 30001")  # 打印换端口建议

---

## 错误 3：ModuleNotFoundError: No module named 'sglang'

**报错信息：**
```
ModuleNotFoundError: No module named 'sglang'
```

**原因分析：**
- SGLang 未安装
- 安装到了不同的 Python 环境（如系统 Python vs conda 环境）
- Jupyter Kernel 使用的 Python 与安装 SGLang 的不一致

**解决方案：**
1. 确认当前 Python 环境
2. 在正确的环境中安装 SGLang

In [ ]:
# ============================================================
# 错误 3 诊断 —— ModuleNotFoundError 排查
# ============================================================

import sys  # 导入系统模块
import subprocess  # 导入子进程模块

print("===== SGLang 安装诊断 =====")  # 打印诊断标题
print()  # 打印空行

# ---------- 检查当前 Python 路径 ----------
print("【1】当前 Python 解释器路径：")  # 打印步骤标题
print(f"  {sys.executable}")  # 打印 Python 可执行文件的完整路径

# ---------- 检查 SGLang 是否安装 ----------
print("\n【2】SGLang 安装状态：")  # 打印步骤标题
check_sglang_installed()  # 调用 SGLang 安装检查函数

# ---------- 列出已安装的相关包 ----------
print("\n【3】已安装的相关包：")  # 打印步骤标题
try:  # 尝试执行 pip list
    result = subprocess.run(  # 运行 pip list 命令
        [sys.executable, "-m", "pip", "list"],  # 使用当前 Python 的 pip
        capture_output=True, text=True  # 捕获输出为文本
    )
    keywords = ["sglang", "torch", "openai", "vllm", "triton"]  # 定义要搜索的关键词
    for line in result.stdout.split("\n"):  # 遍历 pip list 每行
        for kw in keywords:  # 遍历关键词
            if kw in line.lower():  # 如果当前行包含关键词
                print(f"  {line.strip()}")  # 打印匹配的包
                break  # 找到就跳出
except Exception as e:  # 如果发生异常
    print(f"  检查失败: {e}")  # 打印错误信息

# ---------- 安装命令提示 ----------
print("\n【4】安装/重新安装 SGLang：")  # 打印安装提示
print(f"  {sys.executable} -m pip install 'sglang[all]'")  # 打印安装命令

---

## 错误 4：CUDA driver version is insufficient

**报错信息：**
```
RuntimeError: The NVIDIA driver on your system is too old
```

**原因分析：**
- NVIDIA 驱动版本过旧，不支持当前 CUDA 工具包要求
- CUDA 12.1 通常需要驱动版本 >= 525

**解决方案：**
1. 升级 NVIDIA 驱动到最新版
2. 或安装与当前驱动兼容的 PyTorch/CUDA 版本

In [ ]:
# ============================================================
# 错误 4 诊断 —— CUDA driver version 排查
# ============================================================

import subprocess  # 导入子进程模块

print("===== CUDA 驱动版本诊断 =====")  # 打印诊断标题
print()  # 打印空行

# ---------- 检查 NVIDIA 驱动版本 ----------
print("【1】NVIDIA 驱动版本：")  # 打印步骤标题
try:  # 尝试执行 nvidia-smi
    result = subprocess.run(  # 运行 nvidia-smi 查询驱动版本
        ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],  # 查询驱动版本
        capture_output=True, text=True  # 捕获输出
    )
    driver_version = result.stdout.strip()  # 获取驱动版本字符串
    print(f"  当前驱动版本: {driver_version}")  # 打印驱动版本
    try:  # 尝试解析版本号
        major_version = int(driver_version.split(".")[0])  # 提取主版本号
        if major_version >= 525:  # 如果主版本号 >= 525
            print("  [通过] 驱动版本满足 CUDA 12.1 要求")  # 打印满足要求
        else:  # 如果版本不足
            print("  [未通过] 驱动版本可能过旧，建议升级到 525 以上")  # 打印升级建议
    except ValueError:  # 如果版本号解析失败
        print("  无法解析驱动版本号")  # 打印解析失败
except FileNotFoundError:  # 如果 nvidia-smi 不存在
    print("  nvidia-smi 未找到，可能未安装 NVIDIA 驱动")  # 打印未安装提示

# ---------- 检查 CUDA 运行时版本 ----------
print("\n【2】CUDA 运行时版本：")  # 打印步骤标题
try:  # 尝试通过 PyTorch 获取 CUDA 版本
    import torch  # 导入 PyTorch
    print(f"  PyTorch CUDA 版本: {torch.version.cuda}")  # 打印 CUDA 版本
    print(f"  cuDNN 版本: {torch.backends.cudnn.version()}")  # 打印 cuDNN 版本
except (ImportError, AttributeError) as e:  # 如果导入失败
    print(f"  检查失败: {e}")  # 打印失败信息

# ---------- 驱动兼容性参考 ----------
print("\n【3】驱动版本与 CUDA 兼容性参考：")  # 打印兼容性标题
print("  CUDA 12.1 需要驱动 >= 525")  # 打印 CUDA 12.1 要求
print("  CUDA 12.4 需要驱动 >= 550")  # 打印 CUDA 12.4 要求
print("  升级驱动：https://www.nvidia.com/drivers")  # 打印驱动下载链接

---

## 错误 5：Model not found / not supported

**报错信息：**
```
ValueError: Model architecture XXX is not supported.
OSError: XXX is not a local folder and is not a valid model identifier
```

**原因分析：**
- 模型名称/路径拼写错误
- 模型架构不在 SGLang 支持列表中
- 本地模型路径不存在
- 网络无法访问 HuggingFace（国内环境常见）

**解决方案：**
1. 检查模型名称拼写
2. 确认模型架构被 SGLang 支持
3. 使用镜像站或提前下载模型

In [ ]:
# ============================================================
# 错误 5 诊断 —— 模型路径和架构排查
# ============================================================

import os  # 导入操作系统模块

print("===== 模型路径诊断 =====")  # 打印诊断标题
print()  # 打印空行

# ---------- 检查本地模型路径 ----------
model_path = "Qwen/Qwen2.5-0.5B-Instruct"  # 设置默认模型路径

print(f"【1】检查模型路径: {model_path}")  # 打印步骤标题
if os.path.exists(model_path):  # 如果本地路径存在
    print(f"  [通过] 本地路径存在")  # 打印路径存在
    files = os.listdir(model_path)  # 列出目录下的文件
    print(f"  文件数量: {len(files)}")  # 打印文件数量
    key_files = ["config.json", "tokenizer.json", "model.safetensors"]  # 关键文件列表
    for f in key_files:  # 遍历关键文件
        exists = f in files  # 检查文件是否存在
        status = "有" if exists else "无"  # 设置状态
        print(f"    [{status}] {f}")  # 打印检查结果
else:  # 如果本地路径不存在
    print(f"  [信息] 非本地路径，将从 HuggingFace 下载")  # 提示远程下载

# ---------- 常见模型名称参考 ----------
print("\n【2】SGLang 常用模型名称参考：")  # 打印参考标题
models = [  # 定义常用模型列表
    ("Qwen/Qwen2.5-0.5B-Instruct", "0.5B 参数，约 1GB 显存"),  # Qwen 0.5B
    ("Qwen/Qwen2.5-1.5B-Instruct", "1.5B 参数，约 3GB 显存"),  # Qwen 1.5B
    ("Qwen/Qwen2.5-7B-Instruct", "7B 参数，约 14GB 显存"),  # Qwen 7B
    ("meta-llama/Llama-3.1-8B-Instruct", "8B 参数，约 16GB 显存"),  # Llama 3.1
]
for name, desc in models:  # 遍历模型列表
    print(f"  {name:45s} -- {desc}")  # 打印模型名称和描述

# ---------- HuggingFace 镜像提示 ----------
print("\n【3】国内用户下载加速提示：")  # 打印加速提示标题
print("  设置环境变量使用镜像：")  # 打印环境变量提示
print("  export HF_ENDPOINT=https://hf-mirror.com")  # 打印镜像地址

---

## 错误 6：Connection refused（连接被拒绝）

**报错信息：**
```
requests.exceptions.ConnectionError: Connection refused
openai.APIConnectionError: Connection error.
```

**原因分析：**
- SGLang 服务未启动
- 服务正在启动中尚未就绪
- 请求的 host/port 与服务端不一致
- 防火墙阻止了连接

**解决方案：**
1. 确认服务是否已启动
2. 使用健康检查端点验证
3. 检查 host 和 port 是否一致

In [ ]:
# ============================================================
# 错误 6 诊断 —— Connection refused 排查
# ============================================================

import requests  # 导入 HTTP 请求库

print("===== 连接诊断 =====")  # 打印诊断标题
print()  # 打印空行

base_url = "http://127.0.0.1:30000"  # 设置 SGLang 服务的基础 URL

# ---------- 检查端口是否有服务在监听 ----------
print("【1】端口监听检查：")  # 打印步骤标题
check_port_in_use(30000)  # 调用端口检查函数

# ---------- 健康检查 ----------
print("\n【2】SGLang 健康检查：")  # 打印步骤标题
try:  # 尝试发送健康检查请求
    resp = requests.get(f"{base_url}/health", timeout=5)  # 向 /health 端点发送 GET 请求
    if resp.status_code == 200:  # 如果返回 200
        print(f"  [通过] 服务正常运行 (状态码: {resp.status_code})")  # 打印正常提示
    else:  # 如果返回其他状态码
        print(f"  [警告] 服务响应异常 (状态码: {resp.status_code})")  # 打印异常提示
except requests.ConnectionError:  # 如果连接被拒绝
    print("  [未通过] 连接被拒绝 -- 服务可能未启动")  # 打印连接失败
except requests.Timeout:  # 如果请求超时
    print("  [未通过] 请求超时 -- 服务可能正在启动中")  # 打印超时提示

# ---------- 模型信息检查 ----------
print("\n【3】模型信息检查：")  # 打印步骤标题
try:  # 尝试获取模型信息
    resp = requests.get(f"{base_url}/v1/models", timeout=5)  # 向 /v1/models 发送请求
    if resp.status_code == 200:  # 如果请求成功
        models = resp.json()  # 解析 JSON 响应
        print(f"  [通过] 可用模型: {models}")  # 打印可用模型
    else:  # 如果请求失败
        print(f"  [未通过] 获取模型信息失败 (状态码: {resp.status_code})")  # 打印失败
except requests.ConnectionError:  # 如果连接失败
    print("  [未通过] 无法连接服务")  # 打印连接失败
except requests.Timeout:  # 如果超时
    print("  [未通过] 请求超时")  # 打印超时

# ---------- 建议 ----------
print("\n【4】排查建议：")  # 打印建议标题
print("  1. 确认已在终端中执行启动命令")  # 建议确认服务启动
print("  2. 等待 30-120 秒让服务完全启动")  # 建议等待启动完成
print("  3. 检查 base_url 是否正确（默认 http://127.0.0.1:30000）")  # 建议检查地址

---

## 一键诊断脚本

运行以下单元格，自动执行所有诊断检查并生成完整报告。

In [ ]:
# ============================================================
# 一键诊断脚本 —— 自动运行所有检查并生成报告
# ============================================================

import datetime  # 导入日期时间模块
import requests  # 导入 HTTP 请求库

print("=" * 60)  # 打印报告顶部边框
print("           SGLang 环境一键诊断报告")  # 打印报告标题
print("=" * 60)  # 打印报告底部边框
print(f"\n诊断时间: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")  # 打印当前时间
print("=" * 60)  # 打印分隔线

results = {}  # 初始化结果字典

# ---------- 1. Python 版本 ----------
print("\n[1/6] Python 版本检查：")  # 打印第 1 项
results["python"] = check_python_version()  # 执行检查

# ---------- 2. PyTorch 和 CUDA ----------
print("\n[2/6] PyTorch 和 CUDA 检查：")  # 打印第 2 项
results["cuda"] = check_torch_cuda()  # 执行检查

# ---------- 3. SGLang 安装 ----------
print("\n[3/6] SGLang 安装检查：")  # 打印第 3 项
results["sglang"] = check_sglang_installed()  # 执行检查

# ---------- 4. GPU 显存 ----------
print("\n[4/6] GPU 显存检查：")  # 打印第 4 项
check_gpu_memory()  # 执行显存检查

# ---------- 5. 端口检查 ----------
print("\n[5/6] 端口 30000 检查：")  # 打印第 5 项
results["port"] = not check_port_in_use(30000)  # 端口空闲则通过

# ---------- 6. 服务连接检查 ----------
print("\n[6/6] SGLang 服务连接检查：")  # 打印第 6 项
try:  # 尝试发送健康检查
    resp = requests.get("http://127.0.0.1:30000/health", timeout=3)  # 健康检查请求
    results["service"] = resp.status_code == 200  # 状态码 200 则通过
    status_text = "通过" if results["service"] else "未通过"  # 设置状态文字
    print(f"  [{status_text}] 服务状态: {resp.status_code}")  # 打印服务状态
except (requests.ConnectionError, requests.Timeout):  # 如果连接失败
    results["service"] = False  # 标记失败
    print("  [未通过] 服务未运行")  # 打印未运行

# ---------- 汇总报告 ----------
print("\n" + "=" * 60)  # 打印分隔线
print("诊断汇总：")  # 打印汇总标题
print("=" * 60)  # 打印分隔线

labels = {  # 定义各项检查的中文名称
    "python": "Python 版本",  # Python 版本
    "cuda": "PyTorch 和 CUDA",  # PyTorch 和 CUDA
    "sglang": "SGLang 安装",  # SGLang 安装
    "port": "端口 30000 可用",  # 端口可用性
    "service": "SGLang 服务运行",  # 服务运行
}

all_pass = True  # 初始化总体通过标志
for key, label in labels.items():  # 遍历每项检查
    passed = results.get(key, False)  # 获取结果
    status = "通过" if passed else "未通过"  # 设置状态文字
    print(f"  {label:20s} : {status}")  # 打印结果
    if not passed:  # 如果未通过
        all_pass = False  # 总体标志设为 False

print("\n" + "=" * 60)  # 打印分隔线
if all_pass:  # 如果全部通过
    print("所有检查通过！环境就绪，可以正常使用 SGLang。")  # 打印全部通过
else:  # 如果有未通过
    print("存在未通过的检查项，请根据上方诊断信息逐项排查。")  # 打印排查提示

---

## 排查流程图（文字版）

```
遇到 SGLang 错误
    |
    +-- 是安装问题？
    |   +-- 是 -> 检查 Python 环境 -> pip install "sglang[all]"
    |   +-- 否
    |
    +-- 是 CUDA/GPU 问题？
    |   +-- CUDA 不可用 -> 检查驱动版本 -> 升级驱动
    |   +-- OOM -> 降低 mem-fraction / context-length / 用量化模型
    |   +-- 否
    |
    +-- 是网络/连接问题？
    |   +-- Connection refused -> 确认服务已启动 -> 检查 host:port
    |   +-- Address in use -> 终止占用进程 -> 或换端口
    |   +-- 否
    |
    +-- 是模型问题？
        +-- 模型名拼写错误 -> 核对 HuggingFace 模型 ID
        +-- 模型不支持 -> 查看 SGLang 支持列表
        +-- 下载失败 -> 设置 HF_ENDPOINT 镜像
```

---

In [ ]:
# ============================================================
# 清理 —— 本 Notebook 主要是诊断工具，无需特殊清理
# ============================================================

print("本 Notebook 为诊断工具，无需清理服务。")  # 说明无需清理
print("如需停止正在运行的 SGLang 服务，可执行：")  # 打印停止服务提示
print("  kill -9 $(lsof -t -i:30000)")  # 打印终止命令
print("\n清理完成")  # 打印完成提示

## 常见问题排查

### 问题 1：多个错误同时出现

**现象：** 启动 SGLang 时出现多种报错信息。

**排查步骤：**
1. 先停止所有 SGLang 相关进程：`kill -9 $(lsof -t -i:30000)`
2. 重启 Jupyter Kernel 释放已占用的资源
3. 运行一键诊断脚本，确认基础环境无误
4. 如环境有问题，考虑创建新的 conda 环境重新安装：
   ```bash
   conda create -n sglang python=3.10 -y
   conda activate sglang
   pip install "sglang[all]"
   ```

### 问题 2：错误不在本 Notebook 列表中

**现象：** 遇到本 Notebook 未覆盖的错误。

**排查步骤：**
1. 复制完整的报错信息
2. 在 SGLang GitHub Issues 中搜索：https://github.com/sgl-project/sglang/issues
3. 查看 SGLang 官方文档：https://sgl-project.github.io/
4. 确认使用的是最新版本：`pip install --upgrade sglang`

---

### 结语

SGLang 的大多数错误可以通过系统化的排查流程快速定位和解决。建议在遇到问题时，首先运行「一键诊断脚本」获取环境概况，再针对具体错误查找对应的解决方案。